# 03 — Ablation Study

Compare the impact of four design choices, each varied independently from the **ResNet-50 baseline**.

| # | What changes | Baseline | Variant |
|---|---|---|---|
| A | Architecture | ResNet-50 | EfficientNet-B0 |
| B | Backbone freezing | Full fine-tune | Head only |
| C | Augmentation | Strong (crop / flip / jitter) | Minimal (resize + normalize) |
| D | Loss function | Cross-entropy | Focal loss (γ = 2) |

All runs use the same settings: `IMG_SIZE=224`, `BATCH_SIZE=64`, `NUM_EPOCHS=3`, `LR=3e-4`,
same 70 / 15 / 15 stratified split, pretrained ImageNet weights.

In [1]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data  import make_pv_loaders
from src.model import build_model, count_params

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE   = 224
BATCH_SIZE = 64
NUM_EPOCHS = 3
LR         = 3e-4
SEED       = 42

print(f"Device     : {DEVICE}")
print(f"IMG_SIZE={IMG_SIZE}  BATCH_SIZE={BATCH_SIZE}  NUM_EPOCHS={NUM_EPOCHS}  LR={LR}")

Device     : cpu
IMG_SIZE=224  BATCH_SIZE=64  NUM_EPOCHS=3  LR=0.0003


In [2]:
class FocalLoss(nn.Module):
    # Multi-class focal loss (Lin et al., 2017), gamma=2.
    def __init__(self, gamma: float = 2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, logits, targets):
        ce  = F.cross_entropy(logits, targets, reduction="none")
        pt  = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()


def run_epoch(model, loader, criterion, optimizer=None):
    # One train or eval pass. Returns (mean_loss, accuracy).
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss = correct = total = 0
    ctx = torch.enable_grad() if training else torch.inference_mode()
    with ctx:
        for imgs, labels in tqdm(loader, leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs)
            loss   = criterion(logits, labels)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(imgs)
            correct    += (logits.argmax(1) == labels).sum().item()
            total      += len(imgs)
    return total_loss / total, correct / total


# Cache loaders so all variants share the exact same train/val/test split
_LOADERS_CACHE: dict = {}


def get_loaders(train_augment: bool):
    key = "aug" if train_augment else "noaug"
    if key not in _LOADERS_CACHE:
        _LOADERS_CACHE[key] = make_pv_loaders(
            root=ROOT, img_size=IMG_SIZE, batch_size=BATCH_SIZE,
            num_workers=0, seed=SEED, train_augment=train_augment,
        )
    return _LOADERS_CACHE[key]


def train_variant(
    tag: str,
    arch: str = "resnet50",
    freeze_backbone: bool = False,
    train_augment: bool = True,
    criterion: nn.Module = None,
) -> dict:
    # Train one ablation variant for NUM_EPOCHS and return a results dict.
    if criterion is None:
        criterion = nn.CrossEntropyLoss()

    train_loader, val_loader, test_loader, classes = get_loaders(train_augment)
    model = build_model(
        arch=arch, num_classes=len(classes), task="single",
        pretrained=True, freeze_backbone=freeze_backbone,
    ).to(DEVICE)

    p   = count_params(model)
    sep = "=" * 60
    print()
    print(sep)
    print(f"Variant : {tag}")
    print(f"Arch    : {arch}  freeze={freeze_backbone}  aug={train_augment}")
    print(f"Loss    : {criterion.__class__.__name__}")
    print(f"Params  : {p['trainable']:,} trainable / {p['total']:,} total")
    print(sep)

    optimizer = torch.optim.Adam(
        [par for par in model.parameters() if par.requires_grad], lr=LR
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

    best_val_acc = 0.0
    t0 = time.time()
    for epoch in range(1, NUM_EPOCHS + 1):
        tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer)
        vl_loss, vl_acc = run_epoch(model, val_loader,   criterion)
        scheduler.step()
        best_val_acc = max(best_val_acc, vl_acc)
        print(f"  Ep {epoch}/{NUM_EPOCHS} | train {tr_loss:.4f}/{tr_acc:.4f}  val {vl_loss:.4f}/{vl_acc:.4f}")

    _, test_acc = run_epoch(model, test_loader, criterion)
    elapsed = (time.time() - t0) / 60
    print(f"  => test_acc={test_acc:.4f}  ({elapsed:.1f} min)")

    return dict(
        tag=tag, arch=arch, freeze=freeze_backbone,
        augment=train_augment, loss=criterion.__class__.__name__,
        best_val_acc=best_val_acc, test_acc=test_acc, elapsed_min=elapsed,
    )

## Baseline — ResNet-50, full fine-tune, strong aug, cross-entropy

This is the anchor run.  All other variants change exactly **one** factor from this config.

In [3]:
results = []

res = train_variant(
    tag="Baseline (ResNet-50)",
    arch="resnet50",
    freeze_backbone=False,
    train_augment=True,
)
results.append(res)


Variant : Baseline (ResNet-50)
Arch    : resnet50  freeze=False  aug=True
Loss    : CrossEntropyLoss
Params  : 23,585,894 trainable / 23,585,894 total


c:\Users\jake\Documents\2026\CPTS 434\Leaf-Health-Analyzer\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  0%|          | 0/594 [00:00<?, ?it/s]

  0%|          | 0/128 [00:00<?, ?it/s]

  Ep 1/3 | train 0.2597/0.9208  val 0.1291/0.9559


...

  0%|          | 0/594 [00:00<?, ?it/s]

  0%|          | 0/128 [00:00<?, ?it/s]

  Ep 3/3 | train 0.0271/0.9912  val 0.0163/0.9948


  0%|          | 0/128 [00:00<?, ?it/s]

  => test_acc=0.9953  (870.3 min)


## A. Architecture — EfficientNet-B0 vs ResNet-50

EfficientNet-B0 has **4.1 M** parameters vs ResNet-50's **23.6 M** (~6× fewer).
Despite its smaller footprint it was competitive on ImageNet; here we test whether it
matches ResNet-50 on PlantVillage.

In [4]:
res = train_variant(
    tag="A: EfficientNet-B0",
    arch="efficientnet_b0",
    freeze_backbone=False,
    train_augment=True,
)
results.append(res)

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]


Variant : A: EfficientNet-B0
Arch    : efficientnet_b0  freeze=False  aug=True
Loss    : CrossEntropyLoss
Params  : 4,056,226 trainable / 4,056,226 total


  0%|          | 0/594 [00:00<?, ?it/s]

  0%|          | 0/128 [00:00<?, ?it/s]

  Ep 1/3 | train 0.2351/0.9366  val 0.0332/0.9901


  0%|          | 0/594 [00:00<?, ?it/s]

  0%|          | 0/128 [00:00<?, ?it/s]

  Ep 2/3 | train 0.0347/0.9892  val 0.0310/0.9894


  0%|          | 0/594 [00:00<?, ?it/s]

KeyboardInterrupt: 

## B. Backbone freezing — head-only vs full fine-tune

Only the 38-class linear head (77 K params) is trained; the ImageNet backbone is frozen.
Hypothesis: accuracy is meaningfully lower, showing that feature adaptation matters even
on a large pretrained network.

In [ ]:
res = train_variant(
    tag="B: ResNet-50 (frozen backbone)",
    arch="resnet50",
    freeze_backbone=True,
    train_augment=True,
)
results.append(res)

## C. Data augmentation — minimal vs strong

Minimal: resize → centre-crop → normalise only (no random flips, crops, or colour jitter).
Hypothesis: accuracy drops without augmentation — the strong pipeline improves
generalisation even on clean studio images.

In [ ]:
res = train_variant(
    tag="C: ResNet-50 (no augmentation)",
    arch="resnet50",
    freeze_backbone=False,
    train_augment=False,
)
results.append(res)

## D. Loss function — focal loss (γ = 2) vs cross-entropy

Focal loss down-weights easy, well-classified examples and focuses training on hard ones.
PlantVillage is fairly balanced, so any improvement here would indicate the model benefits
from paying more attention to the confused classes seen in `02_evaluate.ipynb`.

In [ ]:
res = train_variant(
    tag="D: ResNet-50 (focal loss, gamma=2)",
    arch="resnet50",
    freeze_backbone=False,
    train_augment=True,
    criterion=FocalLoss(gamma=2.0),
)
results.append(res)

## Results summary

All five variants trained for **3 epochs** from ImageNet-pretrained weights.
Reference: full ResNet-50 baseline trained for **10 epochs** → **98.59 % test accuracy** (see `02_evaluate.ipynb`).

In [ ]:
df_results = pd.DataFrame(results)

# Pretty-print table
display_df = df_results[["tag", "best_val_acc", "test_acc", "elapsed_min"]].copy()
display_df["best_val_acc"] = (display_df["best_val_acc"] * 100).round(2)
display_df["test_acc"]     = (display_df["test_acc"]     * 100).round(2)
display_df["elapsed_min"]  = display_df["elapsed_min"].round(1)
display_df.columns = ["Variant", "Best Val Acc (%)", "Test Acc (%)", "Time (min)"]
print(display_df.to_string(index=False))

# Save CSV
csv_path = ROOT / "models" / "ablation_results.csv"
df_results.to_csv(csv_path, index=False)
print(f"Saved to {csv_path.name}")

# Horizontal bar chart
fig, ax = plt.subplots(figsize=(11, 5))
palette = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0", "#F44336"]
bars = ax.barh(
    df_results["tag"],
    df_results["test_acc"] * 100,
    color=palette[:len(df_results)],
    edgecolor="white",
)
ax.set_xlabel("Test Accuracy (%)", fontsize=11)
ax.set_title("Ablation study: PlantVillage test accuracy (3 epochs each)", fontsize=12)
x_min = df_results["test_acc"].min() * 100 - 3
ax.set_xlim(max(0, x_min), 100)
for bar, val in zip(bars, df_results["test_acc"]):
    ax.text(
        val * 100 + 0.1, bar.get_y() + bar.get_height() / 2,
        f"{val * 100:.2f}%", va="center", fontsize=9,
    )
plt.tight_layout()
plt.savefig(ROOT / "models" / "ablation_results.png", dpi=150)
plt.show()